# Ragas Pipeline

這份 notebook 用來控制 Ragas testset 產生與 eval dataset 建立。主要邏輯已整理在 `generate_ragas_testset.py` 與 `build_ragas_eval_dataset.py`，這裡只設定參數並呼叫函數。


## 操作流程

1. 確認 `.env` 已設定 `OPENAI_API_KEY` 與 `ACCESS_TOKEN`。如果有多組聊天 API token，可再設定 `ACCESS_TOKEN_1`、`ACCESS_TOKEN_2`。
2. 先執行「匯入函數」cell。
3. 到「主要參數」cell 調整 `TESTSET_SIZE`、`DATE_STR`、`VERSION`、`RETRY_FAILED_ONLY`。
4. 一般完整流程請執行「Generate + Build」cell。它會先產生 `ragas_testset.csv`，再接著產生 `ragas_eval_with_response.csv`。
5. 如果已經有 `ragas_testset.csv`，只想補跑 build，請使用最後的「Build only」cell。


## 輸出位置與版本

- Generate 輸出：`eval_ragas_testset/YYYYMMDD/vN/ragas_testset.csv`
- Build 輸出：`eval_ragas_testset/YYYYMMDD/vN/ragas_eval_with_response.csv`
- `DATE_STR = None`：使用今天日期，例如 `20260603`。
- `DATE_STR = "20260603"`：固定輸出到指定日期資料夾。
- `VERSION = None`：自動建立下一個版本。例如當天已有 `v1`，就會建立 `v2`。
- `VERSION = 1` 或 `VERSION = "v1"`：固定使用指定版本資料夾。
- `RETRY_FAILED_ONLY = True`：只重跑 `status == "fail"` 的資料，輸入 CSV 必須已經有 `status` 欄位。


## Docs cache

- Docs cache files are stored in `eval_ragas_testset/docs`.
- Default behavior uses the latest `docs_cache_YYYY-MM-DD.json` in that folder.
- Set `DOCS_CACHE_DATE_STR = "2026-06-03"` or `"20260603"` to use a specific cache file.
- Set `REFRESH_DOCS_CACHE = True` to fetch Firebase again and write today cache into `eval_ragas_testset/docs`.
- Every generate run writes `docs_cache_used.txt` inside that run folder, for example `eval_ragas_testset/YYYYMMDD/vN/docs_cache_used.txt`.


In [ ]:
# 匯入函數
from pathlib import Path

from build_ragas_eval_dataset import build_ragas_eval_dataset
from generate_ragas_testset import generate_and_save_ragas_testset


In [ ]:
# 主要參數
TESTSET_SIZE = 10
RUN_ROOT = Path("eval_ragas_testset")
DOCS_CACHE_ROOT = RUN_ROOT / "docs"

# None = 使用今天日期(folder)；也可以指定字串，例如 "20260603"
DATE_STR = None

# None = 自動建立下一個 vN；也可以指定 1 或 "v1"
VERSION = None

# None = use latest docs_cache_YYYY-MM-DD.json from eval_ragas_testset/docs
# Set "2026-06-03" or "20260603" to use a specific docs cache.
DOCS_CACHE_DATE_STR = "20260604_partial"

# False = use existing cache when available; True = rebuild today cache into eval_ragas_testset/docs
REFRESH_DOCS_CACHE = True

# False = 跑全部；True = 只重跑 status == "fail" 的資料
RETRY_FAILED_ONLY = False

## Generate + Build

一般情況執行下面這個 cell 即可。generate 產生的 CSV 路徑會直接傳給 build，不需要手動指定。

In [ ]:
generate_result = generate_and_save_ragas_testset(
    testset_size=TESTSET_SIZE,
    run_root=RUN_ROOT,
    date_str=DATE_STR,
    version=VERSION,
    docs_cache_root=DOCS_CACHE_ROOT,
    docs_cache_date_str=DOCS_CACHE_DATE_STR,
    refresh_docs_cache=REFRESH_DOCS_CACHE,
)

eval_df, eval_csv = build_ragas_eval_dataset(
    input_csv=generate_result["testset_csv"],
    output_csv=generate_result["run_dir"] / "ragas_eval_with_response.csv",
    retry_failed_only=RETRY_FAILED_ONLY,
)

# print("run_dir:", generate_result["run_dir"])
# print("docs_cache_path:", generate_result["docs_cache_path"])
# print("docs_cache_note:", generate_result["docs_cache_note"])
# print("testset_csv:", generate_result["testset_csv"])
# print("eval_csv:", eval_csv)


## Build only

如果已經有 `ragas_testset.csv`，不需要重新 generate。取消下面 cell 的註解，並把 `INPUT_CSV` 改成要跑的檔案路徑。

In [ ]:
# INPUT_CSV = Path("eval_ragas_testset/20260603/v1/ragas_testset.csv")
# eval_df, eval_csv = build_ragas_eval_dataset(
#     input_csv=INPUT_CSV,
#     output_csv=INPUT_CSV.parent / "ragas_eval_with_response.csv",
#     retry_failed_only=RETRY_FAILED_ONLY,
# )
# print("eval_csv:", eval_csv)
